In [0]:
# ============================================
# CONNECT TO YOUR CATALOG AND SCHEMA
# ============================================

# Set your catalog and schema
spark.sql("USE CATALOG geodata")
spark.sql("USE SCHEMA geo_economics")

print("✓ Connected to geodata.geo_economics")

# Define the path to your uploaded files
volume_path = "/Volumes/geodata/geo_economics/datamain/"

# List files to verify upload
print("\n📁 Your uploaded files:")
files = dbutils.fs.ls(volume_path)
for file in files:
    print(f"   ✓ {file.name}")

# Read all 4 CSV files
print("\n📖 Reading CSV files...")

country_metadata = (spark.read
    .option("header", True)
    .csv(f"{volume_path}country_metadata.csv"))

economic_stress = (spark.read
    .option("header", True)
    .csv(f"{volume_path}economic_stress_score.csv"))

country_indicators = (spark.read
    .option("header", True)
    .csv(f"{volume_path}country_year_indicators.csv"))

indicator_dict = (spark.read
    .option("header", True)
    .csv(f"{volume_path}indicator_dictionary.csv"))

print("✓ All files loaded successfully!")

# Show basic info about each dataset
print("\n📊 Dataset sizes:")
print(f"   Country metadata: {country_metadata.count()} rows")
print(f"   Economic stress scores: {economic_stress.count()} rows")
print(f"   Country indicators: {country_indicators.count()} rows")
print(f"   Indicator dictionary: {indicator_dict.count()} rows")

✓ Connected to geodata.geo_economics

📁 Your uploaded files:
   ✓ country_metadata.csv
   ✓ country_year_indicators.csv
   ✓ economic_stress_score.csv
   ✓ indicator_dictionary.csv

📖 Reading CSV files...
✓ All files loaded successfully!

📊 Dataset sizes:
   Country metadata: 217 rows
   Economic stress scores: 14322 rows
   Country indicators: 14322 rows
   Indicator dictionary: 27 rows


In [0]:
# ============================================
# UNDERSTAND YOUR DATA STRUCTURE
# ============================================

print("=" * 60)
print("DATA SCHEMA ANALYSIS")
print("=" * 60)

# Show column names and sample data for each file
print("\n📋 COUNTRY METADATA columns:")
print(country_metadata.columns)
display(country_metadata.limit(3))

print("\n📋 ECONOMIC STRESS SCORES columns:")
print(economic_stress.columns)
display(economic_stress.limit(3))

print("\n📋 COUNTRY INDICATORS columns:")
print(country_indicators.columns)
display(country_indicators.limit(3))

print("\n📋 INDICATOR DICTIONARY columns:")
print(indicator_dict.columns)
display(indicator_dict.limit(3))

DATA SCHEMA ANALYSIS

📋 COUNTRY METADATA columns:
['country_code', 'country_name', 'iso3', 'region', 'income_group', 'latitude', 'longitude']


country_code,country_name,iso3,region,income_group,latitude,longitude
ABW,Aruba,ABW,Latin America & Caribbean,High income,12.5167,-70.0167
AFG,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,34.5228,69.1761
AGO,Angola,AGO,Sub-Saharan Africa,Lower middle income,-8.81155,13.242



📋 ECONOMIC STRESS SCORES columns:
['country_code', 'country_name', 'year', 'region', 'income_group', 'inflation_score', 'unemployment_score', 'gdp_growth_score', 'income_vulnerability_score', 'food_pressure_score', 'final_economic_stress_score', 'stress_category']


country_code,country_name,year,region,income_group,inflation_score,unemployment_score,gdp_growth_score,income_vulnerability_score,food_pressure_score,final_economic_stress_score,stress_category
ABW,Aruba,1960,Latin America & Caribbean,High income,null,null,null,null,null,null,null
ABW,Aruba,1961,Latin America & Caribbean,High income,null,null,null,null,null,null,null
ABW,Aruba,1962,Latin America & Caribbean,High income,null,null,null,null,null,null,null



📋 COUNTRY INDICATORS columns:
['country_code', 'country_name', 'iso3', 'region', 'income_group', 'year', 'gdp_growth', 'inflation', 'unemployment', 'gdp_per_capita', 'population', 'food_production_index', 'cereal_yield', 'cereal_production_tonnes', 'agricultural_land_pct', 'dietary_energy_supply_adequacy', 'economic_stress_score', 'data_completeness_score']


country_code,country_name,iso3,region,income_group,year,gdp_growth,inflation,unemployment,gdp_per_capita,population,food_production_index,cereal_yield,cereal_production_tonnes,agricultural_land_pct,dietary_energy_supply_adequacy,economic_stress_score,data_completeness_score
ABW,Aruba,ABW,Latin America & Caribbean,High income,1960,null,null,null,null,54922.0,null,null,null,null,null,null,0.0
ABW,Aruba,ABW,Latin America & Caribbean,High income,1961,null,null,null,null,55578.0,null,null,null,11.1111,null,null,0.0
ABW,Aruba,ABW,Latin America & Caribbean,High income,1962,null,null,null,null,56320.0,null,null,null,11.1111,null,null,0.0



📋 INDICATOR DICTIONARY columns:
['column_name', 'indicator_name', 'source', 'source_indicator_code', 'unit', 'description', 'higher_value_interpretation', 'missing_value_notes']


column_name,indicator_name,source,source_indicator_code,unit,description,higher_value_interpretation,missing_value_notes
country_code,Country code,World Bank,ISO3/country id,text,Three-letter country code used for joins.,Identifier only.,Required for published rows.
country_name,Country name,World Bank,country metadata,text,Official country/economy name from World Bank metadata.,Identifier only.,Required for published rows.
iso3,ISO3,World Bank,country metadata,text,ISO3 country code where available.,Identifier only.,Mirrors country_code for World Bank countries.


In [0]:
# ============================================
# DATA CLEANING AND PREPARATION (CORRECTED)
# ============================================

from pyspark.sql.functions import col, when, round as spark_round
from pyspark.sql.types import DoubleType, IntegerType

print("🧹 Cleaning and preparing data...")

# 1. CLEAN ECONOMIC STRESS DATA
economic_stress_clean = economic_stress

# Convert year to integer (if column exists)
if 'year' in economic_stress_clean.columns:
    economic_stress_clean = economic_stress_clean.withColumn("year", col("year").cast(IntegerType()))

# Convert stress scores to double
score_columns = ['inflation_score', 'unemployment_score', 'gdp_growth_score', 
                 'income_vulnerability_score', 'food_pressure_score', 'final_economic_stress_score']

for col_name in score_columns:
    if col_name in economic_stress_clean.columns:
        economic_stress_clean = economic_stress_clean.withColumn(col_name, col(col_name).cast(DoubleType()))

# Filter to years with valid stress scores (1985 onwards)
if 'final_economic_stress_score' in economic_stress_clean.columns and 'year' in economic_stress_clean.columns:
    economic_stress_clean = economic_stress_clean.filter(
        col("final_economic_stress_score").isNotNull() & (col("year") >= 1985)
    )

print(f"✓ Economic stress: {economic_stress_clean.count():,} records")
if economic_stress_clean.count() > 0:
    min_year = economic_stress_clean.agg({'year': 'min'}).collect()[0][0]
    max_year = economic_stress_clean.agg({'year': 'max'}).collect()[0][0]
    num_countries = economic_stress_clean.select('country_code').distinct().count()
    print(f"  Year range: {min_year} - {max_year}")
    print(f"  Countries: {num_countries}")

# 2. CLEAN COUNTRY METADATA
if 'country_code' in country_metadata.columns:
    country_metadata_clean = country_metadata.filter(col("country_code").isNotNull())
    print(f"✓ Country metadata: {country_metadata_clean.count()} countries")
else:
    country_metadata_clean = country_metadata
    print(f"✓ Country metadata: {country_metadata_clean.count()} rows")

# 3. CLEAN COUNTRY INDICATORS
indicator_numeric_cols = ['gdp_growth', 'inflation', 'unemployment', 'gdp_per_capita', 
                          'population', 'food_production_index', 'cereal_yield', 
                          'cereal_production_tonnes', 'agricultural_land_pct', 
                          'data_completeness_score', 'economic_stress_score']

for col_name in indicator_numeric_cols:
    if col_name in country_indicators.columns:
        country_indicators = country_indicators.withColumn(col_name, col(col_name).cast(DoubleType()))

if 'year' in country_indicators.columns:
    country_indicators = country_indicators.withColumn("year", col("year").cast(IntegerType()))

print(f"✓ Country indicators: {country_indicators.count():,} records")
if 'year' in country_indicators.columns and country_indicators.count() > 0:
    min_year = country_indicators.agg({'year': 'min'}).collect()[0][0]
    max_year = country_indicators.agg({'year': 'max'}).collect()[0][0]
    print(f"  Year range: {min_year} - {max_year}")

# 4. CREATE SQL TEMPORARY VIEWS
country_metadata_clean.createOrReplaceTempView("countries")
economic_stress_clean.createOrReplaceTempView("stress")
country_indicators.createOrReplaceTempView("indicators")
indicator_dict.createOrReplaceTempView("indicators_dict")

print("\n✅ SQL views created successfully!")
print("You can now run SQL queries on: countries, stress, indicators, indicators_dict")

# Quick verification
print("\n📊 Quick verification:")
print(f"  Countries view: {spark.sql('SELECT COUNT(*) FROM countries').collect()[0][0]} rows")
print(f"  Stress view: {spark.sql('SELECT COUNT(*) FROM stress').collect()[0][0]} rows")
print(f"  Indicators view: {spark.sql('SELECT COUNT(*) FROM indicators').collect()[0][0]} rows")

🧹 Cleaning and preparing data...
✓ Economic stress: 8,481 records
  Year range: 1985 - 2025
  Countries: 215
✓ Country metadata: 217 countries
✓ Country indicators: 14,322 records
  Year range: 1960 - 2025

✅ SQL views created successfully!
You can now run SQL queries on: countries, stress, indicators, indicators_dict

📊 Quick verification:
  Countries view: 217 rows
  Stress view: 8481 rows
  Indicators view: 14322 rows


In [0]:
# ============================================
# DATA QUALITY ASSESSMENT
# ============================================

print("🔍 Data Quality Check")
print("=" * 50)

# Check for missing values in key columns
print("\nMissing values in STRESS table:")
for col_name in ['country_code', 'year', 'final_economic_stress_score', 'stress_category']:
    if col_name in economic_stress_clean.columns:
        missing = economic_stress_clean.filter(col(col_name).isNull()).count()
        total = economic_stress_clean.count()
        print(f"  {col_name}: {missing} missing ({missing/total*100:.1f}%)")

# Check stress category distribution
print("\nStress Category Distribution:")
category_dist = economic_stress_clean.groupBy("stress_category").count().orderBy("count", ascending=False)
display(category_dist)

# Check region distribution
print("\nRegions in dataset:")
region_dist = country_metadata_clean.groupBy("region").count().orderBy("count", ascending=False)
display(region_dist)

# Check income group distribution
print("\nIncome Groups in dataset:")
income_dist = country_metadata_clean.groupBy("income_group").count().orderBy("count", ascending=False)
display(income_dist)

🔍 Data Quality Check

Missing values in STRESS table:
  country_code: 0 missing (0.0%)
  year: 0 missing (0.0%)
  final_economic_stress_score: 0 missing (0.0%)
  stress_category: 0 missing (0.0%)

Stress Category Distribution:


stress_category,count
Moderate,3777
High,3671
Low,522
Severe,511



Regions in dataset:


region,count
Europe & Central Asia,58
Sub-Saharan Africa,48
Latin America & Caribbean,42
East Asia & Pacific,37
"Middle East, North Africa, Afghanistan & Pakistan",23
South Asia,6
North America,3



Income Groups in dataset:


income_group,count
High income,86
Upper middle income,54
Lower middle income,50
Low income,25
Not classified,2


In [0]:
# ============================================
# CREATE PERMANENT TABLES FOR SQL EDITOR
# ============================================

print("Creating permanent tables for SQL Editor...")

# Save country metadata as permanent table
country_metadata.write.mode("overwrite").saveAsTable("geodata.geo_economics.country_metadata")
print("✓ country_metadata table saved")

# Save economic stress as permanent table
economic_stress.write.mode("overwrite").saveAsTable("geodata.geo_economics.economic_stress")
print("✓ economic_stress table saved")

# Save country indicators as permanent table
country_indicators.write.mode("overwrite").saveAsTable("geodata.geo_economics.country_indicators")
print("✓ country_indicators table saved")

# Save indicator dictionary as permanent table
indicator_dict.write.mode("overwrite").saveAsTable("geodata.geo_economics.indicator_dictionary")
print("✓ indicator_dictionary table saved")

print("\n✅ All tables saved permanently to geodata.geo_economics")
print("\n📊 You can now access these tables in SQL Editor:")
print("   - geodata.geo_economics.country_metadata")
print("   - geodata.geo_economics.economic_stress")
print("   - geodata.geo_economics.country_indicators")
print("   - geodata.geo_economics.indicator_dictionary")

Creating permanent tables for SQL Editor...
✓ country_metadata table saved
✓ economic_stress table saved
✓ country_indicators table saved
✓ indicator_dictionary table saved

✅ All tables saved permanently to geodata.geo_economics

📊 You can now access these tables in SQL Editor:
   - geodata.geo_economics.country_metadata
   - geodata.geo_economics.economic_stress
   - geodata.geo_economics.country_indicators
   - geodata.geo_economics.indicator_dictionary
